# CIS400 — serve a good LLM from Colab for the prompt-injection demo

Use this when you want a **bigger model than the Mac Studio can run**, or when
you are presenting from a machine without a GPU. It runs **vLLM** on the Colab
GPU, exposes an **OpenAI-compatible** `/v1` endpoint, and opens a public
**cloudflared** tunnel. You paste that tunnel URL into the demo's `.env`.

**Cost.** Colab Pro (~USD 10/month) gives an L4 or A100 slice — plenty for a
14B–32B model served to one classroom. The free tier sometimes grants a T4,
which can serve a 7B–8B model. Pay-as-you-go compute units also work.

**Runtime → Change runtime type → GPU** before running. Then run the cells
top to bottom.

In [ ]:
# 1. Check the GPU you were given. This decides which model fits.
!nvidia-smi --query-gpu=name,memory.total --format=csv
# L4 (24GB) / A100 (40-80GB): run a 14B AWQ or 32B AWQ.
# T4 (16GB): stick to a 7-8B model.

In [ ]:
# 2. Install vLLM (OpenAI-compatible server) and grab the cloudflared binary.
!pip -q install 'vllm==0.6.3'
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('ready')

In [ ]:
# 3. Pick the model to fit your GPU. AWQ = 4-bit, fits more on less VRAM.
#   24GB+  ->  Qwen/Qwen2.5-14B-Instruct-AWQ   (recommended: strong tool use)
#   40GB+  ->  Qwen/Qwen2.5-32B-Instruct-AWQ
#   16GB   ->  Qwen/Qwen2.5-7B-Instruct-AWQ
MODEL = 'Qwen/Qwen2.5-14B-Instruct-AWQ'
API_KEY = 'cis400-demo-key'   # the demo must send this; keep it non-empty

import subprocess, sys
server = subprocess.Popen([
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL,
    '--quantization', 'awq',
    '--max-model-len', '8192',
    '--gpu-memory-utilization', '0.92',
    '--enable-auto-tool-choice', '--tool-call-parser', 'hermes',
    '--api-key', API_KEY,
    '--port', '8000',
])
print('vLLM starting (first load pulls weights; wait for the ready line below)...')

In [ ]:
# 4. Wait until vLLM answers, then open the public tunnel.
import time, urllib.request, json, re, subprocess, threading

def wait_ready(timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=2)
            return True
        except Exception:
            time.sleep(3)
    return False

assert wait_ready(), 'vLLM did not come up; check the cell above for errors.'
print('vLLM is up. Opening cloudflared tunnel...')

proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break

print('\n\n==================  PASTE THESE INTO demos/prompt-injection/.env  ==================')
print('LLM_PROFILE=colab')
print(f'LLM_BASE_URL={url}/v1')
print(f'LLM_MODEL={MODEL}')
print(f'LLM_API_KEY={API_KEY}')
print('====================================================================================')
print('\nKeep this Colab tab OPEN for the whole class. Closing it drops the tunnel.')

In [ ]:
# 5. (optional) Sanity check the endpoint from inside Colab.
import urllib.request, json
req = urllib.request.Request(f'{url}/v1/chat/completions',
    data=json.dumps({'model': MODEL, 'messages':[{'role':'user','content':'Say hi in 3 words.'}]}).encode(),
    headers={'Content-Type':'application/json','Authorization':f'Bearer {API_KEY}'})
print(json.loads(urllib.request.urlopen(req).read())['choices'][0]['message']['content'])